# Static Graph Agents — incident-response workflow

This realistic demo validates, saves, retrieves, and runs a v2 static graph through two different branches:

- A production outage that requires current web research.
- An internal follow-up that deliberately skips web research.

It exercises Script, LLM, Conditional, Tool, Agent, Inspector configuration, publishing, and fallback behavior.

## 1. Configuration

Keep credentials in `TEAM_API_KEY` or `AIXPLAIN_API_KEY`; never paste a key here. Set `RUN_LIVE=True` to create and execute resources. Enable `USE_GRAPH_INSPECTOR` only after the engine change that wires graph Inspector nodes is deployed.

In [1]:
import json
import os

from aixplain import Aixplain
from aixplain.v2 import (
    Agent, AgentNode, Condition, ConditionalNode, Edge, Graph, InspectorNode,
    LLMNode, ScriptNode, StaticGraphStrategy, ToolNode,
)

RUN_LIVE = True
USE_GRAPH_INSPECTOR = False
DELETE_DEMO_RESOURCES = True

MODEL_ID = "6a610d0e7dd3d37964ce4c28"
TOOL_ID = "69fb7750f177c224105dabc6"
TOOL_ACTION_NAME = "search"
SUBAGENT_NAME = "Static Graph Incident Analyst"
INSPECTOR_NAME = "Static Graph Response Quality Inspector"

BACKEND_URL = "https://dev-platform-api.aixplain.com"
MODEL_URL = "https://dev-models.aixplain.com/api/v2/execute"
API_KEY = "b65abf3d5e2e1863b888f299261713c6242f0ed5837d5968f0f2395baa370fff"

print("Live execution:", RUN_LIVE)
print("Graph Inspector enabled:", USE_GRAPH_INSPECTOR)
print("API key available:", bool(API_KEY))

/Users/zainaabushaban/Desktop/Work/aiXplain/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Live execution: True
Graph Inspector enabled: False
API key available: True


## 2. Define the incident workflow

The first Script normalizes the ticket and deterministically decides whether current external information is required. An LLM builds a search query. A Conditional routes to Web Search or a no-research Script, both paths merge at a specialist, and a final Conditional publishes a valid briefing or uses the fallback.

In [2]:
prepare_ticket = ScriptNode(
    name="Normalize and classify ticket",
    node_id="prepare_ticket",
    script=(
        "text = input.strip()\n"
        "lower = text.lower()\n"
        "research = ('outage' in lower or 'current' in lower or 'latest' in lower "
        "or 'status page' in lower or 'release' in lower)\n"
        "result = {'clean_ticket': text, 'needs_external_research': research}"
    ),
    script_input_vars=["input"],
)

build_query = LLMNode(
    name="Build an investigation query", node_id="build_query", model=MODEL_ID,
    input_key="clean_ticket", output_key="research_query",
    system_prompt=(
        "Convert the incident report into one concise web-search query. "
        "Include the affected product and symptom. Return only the query."
    ),
    temperature=0, max_tokens=120,
)

research_decision = ConditionalNode(
    name="Decide whether external research is needed",
    node_id="research_decision",
    condition="bool(state.get('needs_external_research'))",
)

web_research = ToolNode(
    name="Research current evidence", node_id="web_research",
    tool_name=TOOL_ACTION_NAME, arg_mapping={"query": "research_query"}, result_key="evidence",
)

skip_research = ScriptNode(
    name="Use internal context only", node_id="skip_research",
    script="result = {'evidence': 'No external research required; use the incident report only.'}",
)

incident_analyst = AgentNode(
    name="Delegate incident analysis", node_id="incident_analyst",
    agent_name=SUBAGENT_NAME,
    agent_prompt=(
        "Incident report: {{clean_ticket}}\n\nAvailable evidence: {{evidence}}\n\n"
        "Prepare a concise briefing with impact, likely cause, immediate actions, "
        "uncertainties, and source URLs when external evidence exists."
    ),
    output_key="draft_briefing",
)

inspect_briefing = InspectorNode(
    name="Inspect briefing quality", node_id="inspect_briefing",
    inspector_type=INSPECTOR_NAME, content_key="draft_briefing", failure_action="warn",
)

briefing_ready = ConditionalNode(
    name="Check for a usable briefing", node_id="briefing_ready",
    condition="bool(state.get('draft_briefing'))",
)

publish_briefing = ScriptNode(
    name="Publish incident briefing", node_id="publish_briefing",
    script="result = {'output': state.get('draft_briefing', '')}",
)

fallback_briefing = ScriptNode(
    name="Return safe fallback", node_id="fallback_briefing",
    script=(
        "result = {'output': 'The incident briefing could not be generated. "
        "Escalate to the on-call incident commander with the original ticket.'}"
    ),
)

## 3. Build and validate both graph variants

`complete_graph` contains every supported node type and proves the full SDK payload. `runnable_graph` includes the graph Inspector only when its engine support is deployed. Agent-level governance remains enabled either way.

In [3]:
common_nodes = [
    prepare_ticket, build_query, research_decision, web_research,
    skip_research, incident_analyst,
]
terminal_nodes = [briefing_ready, publish_briefing, fallback_briefing]

def make_graph(include_graph_inspector):
    after_analysis = inspect_briefing if include_graph_inspector else briefing_ready
    nodes = common_nodes + ([inspect_briefing] if include_graph_inspector else []) + terminal_nodes
    edges = [
        Edge(prepare_ticket, build_query),
        Edge(build_query, research_decision),
        Edge(research_decision, web_research, Condition.is_true("last_condition_bool")),
        Edge(research_decision, skip_research),
        Edge(web_research, incident_analyst),
        Edge(skip_research, incident_analyst),
        Edge(incident_analyst, after_analysis),
    ]
    if include_graph_inspector:
        edges.append(Edge(inspect_briefing, briefing_ready))
    edges.extend([
        Edge(briefing_ready, publish_briefing, Condition.is_true("last_condition_bool")),
        Edge(briefing_ready, fallback_briefing),
    ])
    graph = Graph(entry_point=prepare_ticket, nodes=nodes, edges=edges)
    graph.validate()
    return graph

complete_graph = make_graph(include_graph_inspector=True)
runnable_graph = make_graph(include_graph_inspector=USE_GRAPH_INSPECTOR)

research_edges = [edge for edge in complete_graph.edges if edge.source_id == research_decision.id]
publish_edges = [edge for edge in complete_graph.edges if edge.source_id == briefing_ready.id]
assert research_edges[0].target_id == web_research.id and research_edges[0].condition is not None
assert research_edges[-1].target_id == skip_research.id and research_edges[-1].condition is None
assert publish_edges[-1].target_id == fallback_briefing.id and publish_edges[-1].condition is None

print("Complete graph node types:", [node.TYPE for node in complete_graph.nodes])
print("Runnable graph node types:", [node.TYPE for node in runnable_graph.nodes])
print("Both graph variants are locally valid.")

Complete graph node types: ['script', 'llm', 'conditional', 'tool', 'script', 'agent', 'inspector', 'conditional', 'script', 'script']
Runnable graph node types: ['script', 'llm', 'conditional', 'tool', 'script', 'agent', 'conditional', 'script', 'script']
Both graph variants are locally valid.


## 4. Inspect the exact SDK payload

This offline-safe check confirms graph version, strategy, node fields, ordered conditional edges, and graph-native Inspector serialization.

In [4]:
definition = Agent(
    name="Static graph incident-response definition",
    instructions="Execute the configured incident-response graph.",
    llm=MODEL_ID, graph=complete_graph, strategy=StaticGraphStrategy(max_iterations=30),
)
payload = definition.build_save_payload()
assert payload["graphVersion"] == "1"
assert payload["strategy"]["type"] == "static_graph"
assert payload["graph"]["nodes"]["inspect_briefing"]["type"] == "inspector"
print(json.dumps({key: payload[key] for key in ("graphVersion", "strategy", "graph")}, indent=2))

{
  "graphVersion": "1",
  "strategy": {
    "type": "static_graph",
    "max_iterations": 30
  },
  "graph": {
    "nodes": {
      "prepare_ticket": {
        "id": "prepare_ticket",
        "type": "script",
        "name": "Normalize and classify ticket",
        "script": "text = input.strip()\nlower = text.lower()\nresearch = ('outage' in lower or 'current' in lower or 'latest' in lower or 'status page' in lower or 'release' in lower)\nresult = {'clean_ticket': text, 'needs_external_research': research}",
        "script_input_vars": [
          "input"
        ],
        "sandbox": true
      },
      "build_query": {
        "id": "build_query",
        "type": "llm",
        "name": "Build an investigation query",
        "input_key": "clean_ticket",
        "output_key": "research_query",
        "model": "6a610d0e7dd3d37964ce4c28",
        "temperature": 0,
        "max_tokens": 120,
        "system_prompt": "Convert the incident report into one concise web-search query. Inc

## 5. Create platform resources

The setup verifies the real tool action and its `query` input. The custom Inspector uses the required top-level function name `evaluator_fn`.

In [5]:
aix = tool = specialist = static_agent = None

if RUN_LIVE:
    if not API_KEY:
        raise RuntimeError("Set TEAM_API_KEY or AIXPLAIN_API_KEY first.")

    aix = Aixplain(api_key=API_KEY, backend_url=BACKEND_URL, model_url=MODEL_URL)
    tool = aix.Tool.get(TOOL_ID)
    if TOOL_ACTION_NAME not in tool.actions:
        raise RuntimeError(f"Expected {TOOL_ACTION_NAME!r}; available: {list(tool.actions)}")
    search_action = tool.actions[TOOL_ACTION_NAME]
    if "query" not in search_action.inputs:
        raise RuntimeError(f"The search action has no query input: {list(search_action.inputs)}")
    tool.allowed_actions = [TOOL_ACTION_NAME]

    specialist = aix.Agent(
        name=SUBAGENT_NAME,
        description="Turns incident reports and evidence into actionable operational briefings.",
        instructions=(
            "You are a senior incident analyst. Separate facts from hypotheses, identify immediate "
            "containment steps, call out missing evidence, and cite supplied source URLs."
        ),
        llm=MODEL_ID,
    )

    inspector = aix.Inspector(
        name=INSPECTOR_NAME,
        description="Requires a non-empty briefing with an immediate-action section.",
        severity="medium", targets=["output"], action="continue",
        metric={"function": (
            "def evaluator_fn(text):\n"
            "    value = str(text).strip().lower()\n"
            "    return bool(value) and ('immediate' in value or 'action' in value)\n"
        )},
    )

    static_agent = aix.Agent(
        name="Static graph incident-response coordinator",
        instructions="Execute the configured incident-response graph exactly.",
        llm=MODEL_ID, tools=[tool], agents=[specialist], inspectors=[inspector],
        graph=runnable_graph, strategy=StaticGraphStrategy(max_iterations=30),
    )
    static_agent.save(save_subcomponents=True)
    print("Tool:", tool.name, "/ action:", TOOL_ACTION_NAME)
    print("Saved specialist:", specialist.id)
    print("Saved root Agent:", static_agent.id)
else:
    print("Live setup skipped.")

Tool: Web Search Tool (using Firecrawl) / action: search
Saved specialist: 6a9701ec18fe4f773fd05bc6
Saved root Agent: 6a9701edcbc0254a8d5d7fa8


## 6. Run and verify both real-life branches

The outage case must call Web Search. The internal case must skip it. Once the engine observability branch is deployed, the same checks also verify Script/Conditional steps and exact selected targets.

In [6]:
def summarize_trace(result):
    steps = getattr(result.data, "steps", None) or []
    for index, step in enumerate(steps, start=1):
        unit = step.get("unit") or {}
        print(f"  {index}. {unit.get('name') or step.get('step_id')} [{unit.get('type', 'unknown')}] — {step.get('status', 'unknown')}")
        if unit.get("type") == "node":
            print("     node details:", step.get("output"))
    return steps

def verify_case(agent, label, prompt, expect_web_search, expected_research_target):
    print(f"\n=== {label} ===")
    result = agent.run(prompt)
    output = result.data.output if result.data else None
    print("Status:", result.status)
    print("Output:", output)
    steps = summarize_trace(result)

    assert str(result.status).upper().endswith("SUCCESS"), result.status
    assert output and str(output).strip(), "The workflow returned an empty briefing."
    assert steps, "The backend returned no execution trace."
    assert all(step.get("status") == "completed" for step in steps), steps
    assert all(not step.get("diagnostic_error_codes") for step in steps), steps
    tool_steps = [step for step in steps if (step.get("unit") or {}).get("name") == TOOL_ACTION_NAME]
    assert bool(tool_steps) is expect_web_search, (label, tool_steps)
    subagent_steps = [step for step in steps if (step.get("agent") or {}).get("name") == SUBAGENT_NAME]
    inspector_steps = [step for step in steps if (step.get("unit") or {}).get("name") == INSPECTOR_NAME]
    assert subagent_steps, "The incident-analysis subagent did not appear in the trace."
    assert inspector_steps, "The Agent-level Inspector did not appear in the trace."
    assert inspector_steps[-1].get("action") == "continue", inspector_steps[-1]

    node_steps = {
        (step.get("unit") or {}).get("name"): step for step in steps
        if (step.get("unit") or {}).get("type") == "node"
    }
    if node_steps:
        for required in ("prepare_ticket", "research_decision", "briefing_ready", "publish_briefing"):
            assert required in node_steps, (required, list(node_steps))
        route = node_steps["research_decision"].get("output") or {}
        assert route.get("selected_target") == expected_research_target, route
        publish_route = node_steps["briefing_ready"].get("output") or {}
        assert publish_route.get("selected_target") == "publish_briefing", publish_route
    else:
        print("Local Script/Conditional steps are not exposed by the deployed engine yet.")

    governance = getattr(result.data, "governance", None) if result.data else None
    print("Governance:", governance)
    assert not governance or governance.get("status") in {"ALLOWED", "APPROVED", None}
    return result

if RUN_LIVE:
    fetched = aix.Agent.get(static_agent.id)
    assert fetched.graph.entry_point_id == "prepare_ticket"
    assert fetched.graph.to_dict() == runnable_graph.to_dict()
    assert fetched.strategy.to_dict() == {"type": "static_graph", "max_iterations": 30}
    fetched_types = {node.TYPE for node in fetched.graph.nodes}
    expected_types = {"script", "llm", "conditional", "tool", "agent"}
    if USE_GRAPH_INSPECTOR:
        expected_types.add("inspector")
    assert fetched_types == expected_types, fetched_types

    outage_result = verify_case(
        fetched,
        "Current production outage — research branch",
        ("Production checkout outage: payment confirmations are timing out after today's provider "
         "release. Research the current provider status and prepare an incident briefing."),
        True, "web_research",
    )
    internal_result = verify_case(
        fetched,
        "Internal follow-up — no-research branch",
        ("Prepare a retrospective briefing for yesterday's resolved cache saturation incident. "
         "Use only this ticket: impact lasted 14 minutes, capacity was increased, and no data was lost."),
        False, "skip_research",
    )
else:
    print("Live branch verification skipped.")


=== Current production outage — research branch ===
Status: SUCCESS
Output: ## Incident briefing — Production checkout confirmation timeouts

**Status:** Active / provider health unconfirmed  
**Trigger:** Began after today’s payment-provider release.

### Impact
- Customers are timing out while awaiting payment confirmation.
- A timeout **does not establish that payment failed**; transactions may continue processing after the checkout response expires.
- Key risks: abandoned checkouts, duplicate charges from retries, and orders left pending despite successful payment.

### Likely cause
**Known facts**
- Confirmation delays are occurring after the provider release.
- Payment flows can time out while the provider continues processing asynchronously.
- The provider’s transaction/PaymentIntent status is authoritative; `succeeded` indicates completion.

**Working hypotheses**
1. Provider-side release regression or elevated confirmation latency.
2. Delayed or failed webhook delivery/proces

## 7. Force and verify the final fallback route

This temporary graph makes its condition deterministically false and requires the unconditional edge to return an unmistakable marker.

In [7]:
if RUN_LIVE:
    force_false = ConditionalNode(name="Force fallback", node_id="force_false", condition="False")
    wrong_path = ScriptNode(name="Wrong path", node_id="wrong_path", script="result = {'output': 'WRONG_PATH'}")
    fallback_marker = ScriptNode(
        name="Fallback marker", node_id="fallback_marker",
        script="result = {'output': 'FALLBACK_PATH_CONFIRMED'}",
    )
    smoke_graph = Graph(
        entry_point=force_false, nodes=[force_false, wrong_path, fallback_marker],
        edges=[
            Edge(force_false, wrong_path, Condition.is_true("last_condition_bool")),
            Edge(force_false, fallback_marker),
        ],
    )
    smoke_agent = aix.Agent(
        name="Static graph fallback verification",
        instructions="Execute the configured graph.", llm=MODEL_ID, graph=smoke_graph,
    )
    smoke_agent.save()
    try:
        smoke_result = smoke_agent.run("Verify fallback routing.")
        smoke_output = smoke_result.data.output if smoke_result.data else None
        print("Fallback status:", smoke_result.status)
        print("Fallback output:", smoke_output)
        assert str(smoke_result.status).upper().endswith("SUCCESS"), smoke_result.status
        assert smoke_output == "FALLBACK_PATH_CONFIRMED", smoke_output
        smoke_steps = summarize_trace(smoke_result)
        conditional_steps = [
            step for step in smoke_steps if (step.get("unit") or {}).get("name") == "force_false"
        ]
        if conditional_steps:
            details = conditional_steps[0].get("output") or {}
            assert details.get("selected_target") == "fallback_marker", details
        print("Fallback route confirmed.")
    finally:
        smoke_agent.delete()
else:
    print("Live fallback verification skipped.")

Fallback status: SUCCESS
Fallback output: FALLBACK_PATH_CONFIRMED
  1. force_false [node] — completed
     node details: {'selected_target': 'fallback_marker', 'condition_result': False, 'state_keys': ['force_false_result', 'last_condition_result', 'last_condition_bool']}
  2. fallback_marker [node] — completed
     node details: {'state_keys': ['output']}
Fallback route confirmed.


## 8. Cleanup

Cleanup removes only the root Agent and specialist created here. The existing Web Search tool is never deleted.

In [8]:
if RUN_LIVE and DELETE_DEMO_RESOURCES:
    if static_agent is not None:
        static_agent.delete()
    if specialist is not None:
        specialist.delete()
    print("Deleted the demo root Agent and specialist.")
else:
    print("Cleanup skipped.")

Deleted the demo root Agent and specialist.
